# 웰니스 앱 리텐션 분석용 전처리

조원 전처리 코드를 한 번 정리한 버전입 
**분석에 필요한 파생컬럼과 플래그를 만드는 것**을 목표로 합니다.

## preprocessed.ipynb 기준에서 정리한 부분
- `is_log_anomaly`는  `is_log_issue_period`로 이름을 바꿔 저장했습니다.
- 첫 이벤트 소요시간이 음수인 사용자는 삭제하지 않고 `is_first_event_before_signup` 플래그로 표시합니다.
- `first_event_time`, `first_event_elapsed_hours`, `avg_events_per_session`은 임시 계산에서 끝내지 않고 저장용 컬럼으로 남깁니다.

## 주요 방향
- 기본 타입 정리
- 날짜/요일/월 파생컬럼 생성
- 로그 장애 기간 플래그 생성
- 결측치/이상치 플래그 생성
- 유저 단위 분석용 파생컬럼 생성

## 최종 저장 파일
1. `01_User_Profile_preprocessed.csv`
2. `02_Event_Log_preprocessed.csv`

# 추가된 파생컬럼 및 플래그 설명

1. 날짜 분석용 컬럼 추가
    - 가입월, 가입요일, 이벤트월, 이벤트요일, 이벤트시간대
2. 초기 행동 흐름 분석용 컬럼 추가
    - 첫 이벤트 시간
    - 가입 후 첫 이벤트까지 걸린 시간
    - 첫 앱실행 시간
    - 가입 후 앱실행까지 걸린 시간
    - 온보딩 완료 시간
    - 앱실행 후 온보딩 완료까지 걸린 시간
3. 사용자 활동량 분석용 컬럼 추가
    - 사용자별 전체 이벤트 수
    - 세션당 평균 이벤트 수
4. 데이터 품질 확인용 플래그 추가
    - 로그 장애 기간 여부
    - 이벤트 로그 없는 사용자 여부
    - 이벤트 타입 결측 여부
    - 가입일보다 첫 이벤트가 빠른 사용자 여부

# 라이브러리 및 데이터 호출

In [7]:
# 0. 라이브러리 호출

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# 1. 프로젝트 경로 설정
# 프로젝트 최상위 폴더입니다.
# 본인 PC에서 GitHub 프로젝트가 있는 위치로 바꾸면 됩니다.
ROOT = Path(r"C:\Users\milkl\OneDrive\문서\웰니스 공모전\wellness analyze\Anti-Churn-Committee")

# 원본 데이터가 들어있는 폴더
DATA_RAW_DIR = ROOT / "data" / "raw"

# 전처리 결과를 저장할 폴더
DATA_PROCESSED_DIR = ROOT / "data" / "processed"

# 원본 파일명입니다.
# data/raw 폴더 안에 이 이름으로 파일이 있어야 합니다.
USER_PROFILE_FILE = "01_User_Profile.csv"
EVENT_LOG_FILE = "02_Event_Log.csv"

# 실제로 읽어올 파일 경로를 만듭니다.
USER_PROFILE_PATH = DATA_RAW_DIR / USER_PROFILE_FILE
EVENT_LOG_PATH = DATA_RAW_DIR / EVENT_LOG_FILE

print("USER_PROFILE_PATH:", USER_PROFILE_PATH)
print("EVENT_LOG_PATH   :", EVENT_LOG_PATH)
print("저장 폴더        :", DATA_PROCESSED_DIR)

USER_PROFILE_PATH: C:\Users\milkl\OneDrive\문서\웰니스 공모전\wellness analyze\Anti-Churn-Committee\data\raw\01_User_Profile.csv
EVENT_LOG_PATH   : C:\Users\milkl\OneDrive\문서\웰니스 공모전\wellness analyze\Anti-Churn-Committee\data\raw\02_Event_Log.csv
저장 폴더        : C:\Users\milkl\OneDrive\문서\웰니스 공모전\wellness analyze\Anti-Churn-Committee\data\processed


In [8]:
# 원본 데이터 읽기
df_raw01 = pd.read_csv(USER_PROFILE_PATH)   # 사용자 프로필 데이터
df_raw02 = pd.read_csv(EVENT_LOG_PATH)      # 이벤트 로그 데이터

print("df_raw01 shape:", df_raw01.shape)
print("df_raw02 shape:", df_raw02.shape)

display(df_raw01.head())
display(df_raw02.head())

df_raw01 shape: (12500, 6)
df_raw02 shape: (1757262, 5)


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자
0,U0000001,2025-01-25,오가닉,iOS,True,NaN
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24
2,U0000003,2025-05-14,오가닉,iOS,False,NaN
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaN
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaN


,User_ID,Event_Time,Event_Type,Session_ID,알림_유형
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,NaN
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,NaN
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,NaN
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,NaN
4,U0000001,2025-01-25 20:30:00,알림수신,NaN,광고성


In [9]:
# ============================================================
# 원본 데이터 구조 확인
# ============================================================

print("User_Profile 컬럼")
print(df_raw01.columns.tolist())

print("\nEvent_Log 컬럼")
print(df_raw02.columns.tolist())

print("\nUser_Profile 결측치")
display(df_raw01.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nEvent_Log 결측치")
display(df_raw02.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\n중복 확인")
print("User_Profile 전체 중복 행:", df_raw01.duplicated().sum())
print("Event_Log 전체 중복 행    :", df_raw02.duplicated().sum())
print("User_Profile User_ID 중복 :", df_raw01.duplicated(subset=["User_ID"]).sum())

User_Profile 컬럼
['User_ID', '가입일자', '가입경로', '기기', '알림수신동의여부', '알림수신동의_변경일자']

Event_Log 컬럼
['User_ID', 'Event_Time', 'Event_Type', 'Session_ID', '알림_유형']

User_Profile 결측치


,missing_count
알림수신동의_변경일자,10524
가입경로,137
기기,121
알림수신동의여부,116
가입일자,0
User_ID,0



Event_Log 결측치


,missing_count
알림_유형,1538380
Session_ID,241502
Event_Type,26456
User_ID,0
Event_Time,0



중복 확인
User_Profile 전체 중복 행: 0
Event_Log 전체 중복 행    : 0
User_Profile User_ID 중복 : 0


# 기본 전처리

In [10]:
# ============================================================
# 1. 기본 전처리
# - 원본 데이터 복사
# - 컬럼명 앞뒤 공백 제거
# - 문자열 값 앞뒤 공백 제거
# - 빈 문자열을 결측치로 처리
# - 날짜 컬럼 타입 변환
# ============================================================
user_profile = df_raw01.copy()
event_log = df_raw02.copy()

# 컬럼명 앞뒤 공백 제거
user_profile.columns = user_profile.columns.str.strip()
event_log.columns = event_log.columns.str.strip()

# 문자열 값 앞뒤 공백 제거
user_text_cols = user_profile.select_dtypes(include=["object", "string"]).columns
event_text_cols = event_log.select_dtypes(include=["object", "string"]).columns

# lambda는 각 컬럼에 같은 처리를 반복 적용하기 위해 사용
user_profile[user_text_cols] = user_profile[user_text_cols].apply(
    lambda col: col.astype("string").str.strip()
)
event_log[event_text_cols] = event_log[event_text_cols].apply(
    lambda col: col.astype("string").str.strip()
)

# 빈 문자열은 결측치로 처리
user_profile = user_profile.replace("", pd.NA)
event_log = event_log.replace("", pd.NA)

# 두 데이터의 User_ID 타입 통일
user_profile["User_ID"] = user_profile["User_ID"].astype("string")
event_log["User_ID"] = event_log["User_ID"].astype("string")

# 날짜 컬럼 타입 변환
user_profile["가입일자"] = pd.to_datetime(
    user_profile["가입일자"],
    errors="coerce"
)

user_profile["알림수신동의_변경일자"] = pd.to_datetime(
    user_profile["알림수신동의_변경일자"],
    errors="coerce"
)

event_log["Event_Time"] = pd.to_datetime(
    event_log["Event_Time"],
    errors="coerce"
)

print("기본 전처리 완료")
display(user_profile.head())
display(event_log.head())


기본 전처리 완료


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자
0,U0000001,2025-01-25,오가닉,iOS,True,NaT
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24
2,U0000003,2025-05-14,오가닉,iOS,False,NaT
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaT
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaT


,User_ID,Event_Time,Event_Type,Session_ID,알림_유형
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,<NA>
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,<NA>
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,<NA>
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,<NA>
4,U0000001,2025-01-25 20:30:00,알림수신,<NA>,광고성


# 파생컬럼 생성

In [11]:
# ============================================================
# 1. 날짜/요일 파생컬럼 생성
# - 가입일, 가입월, 가입요일
# - 이벤트일, 이벤트월, 이벤트요일, 이벤트시간대
# ============================================================

weekday_map = {
    0: "월",
    1: "화",
    2: "수",
    3: "목",
    4: "금",
    5: "토",
    6: "일"
}

# User_Profile 날짜 파생컬럼
user_profile["가입일"] = user_profile["가입일자"].dt.date
user_profile["가입월"] = user_profile["가입일자"].dt.to_period("M").astype(str)
user_profile["가입요일번호"] = user_profile["가입일자"].dt.dayofweek
user_profile["가입요일"] = user_profile["가입요일번호"].map(weekday_map)

# Event_Log 날짜 파생컬럼
event_log["이벤트일"] = event_log["Event_Time"].dt.date
event_log["이벤트월"] = event_log["Event_Time"].dt.to_period("M").astype(str)
event_log["이벤트요일번호"] = event_log["Event_Time"].dt.dayofweek
event_log["이벤트요일"] = event_log["이벤트요일번호"].map(weekday_map)
event_log["이벤트시간대"] = event_log["Event_Time"].dt.hour

print("날짜 파생컬럼 생성 완료")
display(user_profile.head())
display(event_log.head())

날짜 파생컬럼 생성 완료


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자,가입일,가입월,가입요일번호,가입요일
0,U0000001,2025-01-25,오가닉,iOS,True,NaT,2025-01-25,2025-01,5,토
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24,2025-05-06,2025-05,1,화
2,U0000003,2025-05-14,오가닉,iOS,False,NaT,2025-05-14,2025-05,2,수
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaT,2025-02-23,2025-02,6,일
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaT,2025-02-18,2025-02,1,화


,User_ID,Event_Time,Event_Type,Session_ID,알림_유형,이벤트일,이벤트월,이벤트요일번호,이벤트요일,이벤트시간대
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,<NA>,2025-01-25,2025-01,5,토,7
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,<NA>,2025-01-25,2025-01,5,토,7
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,<NA>,2025-01-25,2025-01,5,토,7
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,<NA>,2025-01-25,2025-01,5,토,7
4,U0000001,2025-01-25 20:30:00,알림수신,<NA>,광고성,2025-01-25,2025-01,5,토,20


# 플래그 생성

삭제하거나 제외하지 않고, 분석할 때 구분할 수 있도록 플래그만 만듭니다.


In [12]:
# ============================================================
# 1. 로그 수집 장애 기간 플래그
# - 2025년 3월 10일 ~ 2025년 3월 14일
# ============================================================

LOG_ISSUE_START = pd.Timestamp("2025-03-10")
LOG_ISSUE_END = pd.Timestamp("2025-03-14 23:59:59")

# 장애 기간에 발생한 이벤트 여부
# preprocessed.ipynb의 is_log_anomaly와 같은 목적의 플래그
# 저장 컬럼명은 의미가 더 잘 보이도록 is_log_issue_period로 사용

event_log["is_log_issue_period"] = event_log["Event_Time"].between(
    LOG_ISSUE_START,
    LOG_ISSUE_END
)

print("로그 장애 기간 이벤트 수:", event_log["is_log_issue_period"].sum())

로그 장애 기간 이벤트 수: 20862


# 추가한 내용

## 파생컬럼

In [13]:
notification_events = ["알림수신", "알림오픈"]

### 첫 행동 시점 관련 파생컬럼

In [14]:
# ------------------------------
# 유저별 첫 이벤트 시간
# - 가입 후 첫 이벤트까지 걸린 시간
# ------------------------------
first_event = (
    event_log
    .groupby("User_ID")["Event_Time"]
    .min()
    .reset_index()
)
first_event.columns = ["User_ID", "first_event_time"]

user_profile = user_profile.merge(
    first_event,
    on="User_ID",
    how="left"
)

# 가입 후 첫 이벤트까지 몇 시간이 걸렸는지 계산
# total_seconds()는 시간 차이를 초 단위로 계산, 600으로 나누면 시간 단위가 됨
user_profile["first_event_elapsed_hours"] = (
    user_profile["first_event_time"] - user_profile["가입일자"]
).dt.total_seconds() / 3600

In [15]:
# ------------------------------
# 유저별 첫 앱실행 시간
# - 가입 후 첫 앱실행까지 걸린 시간
# ------------------------------
first_app_launch = (
    event_log[event_log["Event_Type"] == "앱실행"]
    .groupby("User_ID")["Event_Time"]
    .min()
    .reset_index()
)
first_app_launch.columns = ["User_ID", "first_app_launch_time"]

user_profile = user_profile.merge(
    first_app_launch,
    on="User_ID",
    how="left"
)

# 가입 후 첫 앱실행까지 걸린 시간
# total_seconds()는 시간 차이를 초 단위로 계산, 600으로 나누면 시간 단위가 됨
user_profile["app_launch_elapsed_hours"] = (
    user_profile["first_app_launch_time"] - user_profile["가입일자"]
).dt.total_seconds() / 3600


In [16]:
# ------------------------------
# 유저별 첫 온보딩 완료 시간
# - 가입 후 온보딩 완료까지 걸린 시간
# - 앱실행 후 온보딩 완료까지 걸린 시간
# ------------------------------
first_onboarding = (
    event_log[event_log["Event_Type"] == "온보딩_완료"]
    .groupby("User_ID")["Event_Time"]
    .min()
    .reset_index()
)
first_onboarding.columns = ["User_ID", "onboarding_completed_time"]

user_profile = user_profile.merge(
    first_onboarding,
    on="User_ID",
    how="left"
)

# 가입 후 온보딩 완료까지 걸린 시간
# total_seconds()는 시간 차이를 초 단위로 계산, 3600으로 나누면 시간 단위가 됨
user_profile["onboarding_elapsed_hours"] = (
    user_profile["onboarding_completed_time"] - user_profile["가입일자"]
).dt.total_seconds() / 3600


# 첫 앱실행 후 온보딩 완료까지 걸린 시간
# total_seconds()는 시간 차이를 초 단위로 계산, 3600으로 나누면 시간 단위가 됨
user_profile["app_launch_to_onboarding_hours"] = (
    user_profile["onboarding_completed_time"]
    - user_profile["first_app_launch_time"]
).dt.total_seconds() / 3600

# 온보딩 완료 시간이 있으면 온보딩을 완료한 사용자
user_profile["is_onboarding_completed"] = (
    user_profile["onboarding_completed_time"].notna()
)

In [17]:
print("첫 행동 시점 파생컬럼 생성 완료")
display(
    user_profile[
        [
            "User_ID",
            "first_event_time",
            "first_event_elapsed_hours",
            "first_app_launch_time",
            "app_launch_elapsed_hours",
            "onboarding_completed_time",
            "onboarding_elapsed_hours",
            "is_onboarding_completed"
        ]
    ].head()
)

첫 행동 시점 파생컬럼 생성 완료


,User_ID,first_event_time,first_event_elapsed_hours,first_app_launch_time,app_launch_elapsed_hours,onboarding_completed_time,onboarding_elapsed_hours,is_onboarding_completed
0,U0000001,2025-01-25 07:25:45,7.429167,2025-01-25 07:25:45,7.429167,2025-01-25 07:26:15,7.437500,True
1,U0000002,2025-05-06 16:23:12,16.386667,2025-05-06 16:23:12,16.386667,NaT,NaN,False
2,U0000003,2025-05-14 11:09:58,11.166111,2025-05-14 11:09:58,11.166111,NaT,NaN,False
3,U0000004,2025-02-23 07:15:35,7.259722,2025-02-23 07:15:35,7.259722,NaT,NaN,False
4,U0000005,2025-02-18 12:50:01,12.833611,2025-02-18 12:52:37,12.876944,2025-02-18 12:53:07,12.885278,True


###  행동량 관련 파생컬럼

In [18]:
# ------------------------------
# 유저별 전체 이벤트 수
# ------------------------------
user_event_count = (
    event_log
    .groupby("User_ID")
    .size()
    .reset_index(name="event_count_total")
)

user_profile = user_profile.merge(
    user_event_count,
    on="User_ID",
    how="left"
)

# 이벤트가 없는 사용자는 NaN으로 들어오므로 0으로 변경
user_profile["event_count_total"] = (
    user_profile["event_count_total"]
    .fillna(0)
    .astype(int)
)

In [19]:
# ------------------------------
# 세션당 평균 이벤트 수
# ------------------------------
# 알림 이벤트는 앱 밖 이벤트이므로 세션 평균 계산에서는 제외
event_no_alarm = event_log[
    ~event_log["Event_Type"].isin(notification_events)
].copy()

# 사용자별, 세션별 이벤트 수를 계산
session_event_count = (
    event_no_alarm
    .groupby(["User_ID", "Session_ID"])
    .size()
    .reset_index(name="session_event_count")
)

# 사용자별 세션당 평균 이벤트 수를 계산
user_session_avg = (
    session_event_count
    .groupby("User_ID")["session_event_count"]
    .mean()
    .reset_index()
)
user_session_avg.columns = ["User_ID", "avg_events_per_session"]


# 사용자 프로필에 세션당 평균 이벤트 수를 조인
user_profile = user_profile.merge(
    user_session_avg,
    on="User_ID",
    how="left"
)


# 세션이 없는 사용자는 0으로 처리
user_profile["avg_events_per_session"] = (
    user_profile["avg_events_per_session"]
    .fillna(0)
    .round(2)
)

In [20]:
print("행동량 관련 파생컬럼 생성 완료")

display(
    user_profile[
        [
            "User_ID",
            "event_count_total",
            "avg_events_per_session"
        ]
    ].head()
)

행동량 관련 파생컬럼 생성 완료


,User_ID,event_count_total,avg_events_per_session
0,U0000001,515,2.07
1,U0000002,55,2.62
2,U0000003,3,1.50
3,U0000004,89,1.79
4,U0000005,442,1.85


## 플래그

### User_Profile 기준 기본 플래그

In [21]:
# ============================================================
# 추가 플래그 생성
# - User_Profile 기준 기본 플래그
# ============================================================

# 장애 기간에 가입한 사용자 여부
user_profile["is_signup_log_issue_period"] = user_profile["가입일자"].between(
    LOG_ISSUE_START,
    LOG_ISSUE_END
)

# 프로필에는 있지만 이벤트 로그에는 없는 사용자
# 이 사용자를 무조건 삭제하거나 무활동 사용자로 단정하지 않고, 분석할 때 따로 볼 수 있게 플래그 생성
event_user_ids = set(event_log["User_ID"])
user_profile["is_no_event_user"] = ~user_profile["User_ID"].isin(event_user_ids)

# 알림 수신 동의 변경 여부
# 동의 상태가 한 번이라도 바뀐 사용자
user_profile["is_notification_changed"] = (
    user_profile["알림수신동의_변경일자"].notna()
)

# 온보딩 완료 시간이 있으면 온보딩을 완료한 사용자
user_profile["is_onboarding_completed"] = (
    user_profile["onboarding_completed_time"].notna()
)

print("User_Profile 기준 기본 플래그 생성 완료")
print("로그 장애 기간 가입자 수:", user_profile["is_signup_log_issue_period"].sum())
print("이벤트 로그 없는 사용자 수:", user_profile["is_no_event_user"].sum())
print("알림 수신 동의 변경 사용자 수:", user_profile["is_notification_changed"].sum())
print("온보딩 완료 사용자 수:", user_profile["is_onboarding_completed"].sum())

display(
    user_profile[
        [
            "User_ID",
            "is_signup_log_issue_period",
            "is_no_event_user",
            "is_notification_changed",
            "is_onboarding_completed"
        ]
    ].head()
)

User_Profile 기준 기본 플래그 생성 완료
로그 장애 기간 가입자 수: 341
이벤트 로그 없는 사용자 수: 47
알림 수신 동의 변경 사용자 수: 1976
온보딩 완료 사용자 수: 5719


,User_ID,is_signup_log_issue_period,is_no_event_user,is_notification_changed,is_onboarding_completed
0,U0000001,False,False,False,True
1,U0000002,False,False,True,False
2,U0000003,False,False,False,False
3,U0000004,False,False,False,False
4,U0000005,False,False,False,True


### User_Profile 기준 이상치 플래그

In [22]:
# ============================================================
# 추가 플래그 생성
# - User_Profile 기준 데이터 확인 플래그
# ============================================================

# 첫 이벤트 시간이 가입일보다 빠른 경우 확인용 플래그
user_profile["is_first_event_before_signup"] = (
    user_profile["first_event_elapsed_hours"] < 0
)

print("User_Profile 기준 데이터 확인 플래그 생성 완료")
print("가입일보다 첫 이벤트가 빠른 사용자 수:", user_profile["is_first_event_before_signup"].sum())

display(
    user_profile[
        [
            "User_ID",
            "first_event_time",
            "first_event_elapsed_hours",
            "is_first_event_before_signup"
        ]
    ].head()
)



User_Profile 기준 데이터 확인 플래그 생성 완료
가입일보다 첫 이벤트가 빠른 사용자 수: 0


,User_ID,first_event_time,first_event_elapsed_hours,is_first_event_before_signup
0,U0000001,2025-01-25 07:25:45,7.429167,False
1,U0000002,2025-05-06 16:23:12,16.386667,False
2,U0000003,2025-05-14 11:09:58,11.166111,False
3,U0000004,2025-02-23 07:15:35,7.259722,False
4,U0000005,2025-02-18 12:50:01,12.833611,False


### Event_Log 기준 플래그

In [23]:
# ============================================================
# 추가 플래그 생성
# - Event_Log 기준 플래그
# ============================================================

# Event_Type 결측 확인 플래그
event_log["is_event_type_missing"] = event_log["Event_Type"].isna()

print("Event_Log 기준 플래그 생성 완료")
print("Event_Type 결측 이벤트 수:", event_log["is_event_type_missing"].sum())

display(
    event_log[
        [
            "User_ID",
            "Event_Type",
            "is_event_type_missing"
        ]
    ].head()
)

Event_Log 기준 플래그 생성 완료
Event_Type 결측 이벤트 수: 26456


,User_ID,Event_Type,is_event_type_missing
0,U0000001,앱실행,False
1,U0000001,온보딩_완료,False
2,U0000001,챌린지_탐색,False
3,U0000001,챌린지참여,False
4,U0000001,알림수신,False


## 임의 추가

In [24]:
# 이벤트 로그가 없는 사용자 확인

# is_no_event_user가 True인 사용자만 따로 범
no_event_users = user_profile[user_profile["is_no_event_user"]].copy()

print("이벤트 로그 없는 사용자 수:", len(no_event_users))

if len(no_event_users) > 0:
    print("\n[이벤트 로그 없는 사용자 가입월]")
    display(
        no_event_users
        .groupby("가입월")
        .size()
        .reset_index(name="사용자수")
    )

    print("\n[이벤트 로그 없는 사용자 중 로그 장애 기간 가입자 수]")
    print(no_event_users["is_signup_log_issue_period"].sum())

이벤트 로그 없는 사용자 수: 47

[이벤트 로그 없는 사용자 가입월]


,가입월,사용자수
0,2025-03,47



[이벤트 로그 없는 사용자 중 로그 장애 기간 가입자 수]
47


In [25]:
# 가입 → 앱실행 → 온보딩 완료 퍼널 확인
# - 초기 이탈 위치를 보기 위한 기본 확인용
funnel_by_month = (
    user_profile
    .groupby("가입월")
    .agg(
        total_users=("User_ID", "count"),
        app_launch_users=("first_app_launch_time", "count"),
        onboarding_users=("onboarding_completed_time", "count")
    )
    .reset_index()
)

# 앱실행 전환율 = 앱실행 사용자 수 / 전체 가입자 수
funnel_by_month["app_launch_rate"] = (
    funnel_by_month["app_launch_users"]
    / funnel_by_month["total_users"]
).round(4)

# 온보딩 완료율 = 온보딩 완료 사용자 수 / 전체 가입자 수
funnel_by_month["onboarding_rate"] = (
    funnel_by_month["onboarding_users"]
    / funnel_by_month["total_users"]
).round(4)

print("[가입월별 가입 → 앱실행 → 온보딩 완료 퍼널]")
display(funnel_by_month)

[가입월별 가입 → 앱실행 → 온보딩 완료 퍼널]


,가입월,total_users,app_launch_users,onboarding_users,app_launch_rate,onboarding_rate
0,2025-01,2124,2122,999,0.9991,0.4703
1,2025-02,4384,4382,2042,0.9995,0.4658
2,2025-03,2122,2074,899,0.9774,0.4237
3,2025-04,2082,2081,941,0.9995,0.4520
4,2025-05,1788,1788,838,1.0000,0.4687


# 저장 전 간단 점검

In [26]:
print("[User_Profile]")
print("전체 사용자 수:", len(user_profile))
print("이벤트 없는 사용자 수:", user_profile["is_no_event_user"].sum())
print("온보딩 완료 사용자 수:", user_profile["is_onboarding_completed"].sum())
print()

print("[Event_Log]")
print("전체 이벤트 수:", len(event_log))
print("로그 장애 기간 이벤트 수:", event_log["is_log_issue_period"].sum())
print("Event_Type 결측 이벤트 수:", event_log["is_event_type_missing"].sum())
print()

print("[월별 가입자 수]")
display(
    user_profile
    .groupby("가입월")
    .size()
    .reset_index(name="가입자수")
)

print("[가입월 x 가입경로]")
display(
    user_profile
    .groupby(["가입월", "가입경로"])
    .size()
    .reset_index(name="가입자수")
)

[User_Profile]
전체 사용자 수: 12500
이벤트 없는 사용자 수: 47
온보딩 완료 사용자 수: 5719

[Event_Log]
전체 이벤트 수: 1757262
로그 장애 기간 이벤트 수: 20862
Event_Type 결측 이벤트 수: 26456

[월별 가입자 수]


,가입월,가입자수
0,2025-01,2124
1,2025-02,4384
2,2025-03,2122
3,2025-04,2082
4,2025-05,1788


[가입월 x 가입경로]


,가입월,가입경로,가입자수
0,2025-01,오가닉,966
1,2025-01,퍼포먼스광고,1134
2,2025-02,오가닉,1943
3,2025-02,퍼포먼스광고,2390
4,2025-03,오가닉,904
5,2025-03,퍼포먼스광고,1204
6,2025-04,오가닉,916
7,2025-04,퍼포먼스광고,1142
8,2025-05,오가닉,782
9,2025-05,퍼포먼스광고,982


# 저장

In [27]:
# ============================================================
# 저장용 컬럼명 변경
# - 저장할 때만 영어 소문자 형식으로 변경
# - 요일 번호 컬럼은 요일명을 만들기 위한 중간 컬럼이므로 저장 전 제거
# ============================================================

column_rename_map = {
    "User_ID": "user_id",
    "가입일자": "signup_date",
    "가입경로": "signup_channel",
    "기기": "device",
    "알림수신동의여부": "notification_agreed",
    "알림수신동의_변경일자": "notification_changed_date",
    "가입일": "signup_day",
    "가입월": "signup_month",
    "가입요일": "signup_weekday",
    "Event_Time": "event_time",
    "Event_Type": "event_type",
    "Session_ID": "session_id",
    "알림_유형": "notification_type",
    "이벤트일": "event_date",
    "이벤트월": "event_month",
    "이벤트요일": "event_weekday",
    "이벤트시간대": "event_hour"
}

drop_columns_for_save = [
    "가입요일번호",
    "이벤트요일번호"
]

user_profile_save = (
    user_profile
    .drop(columns=drop_columns_for_save, errors="ignore")
    .rename(columns=column_rename_map)
)

event_log_save = (
    event_log
    .drop(columns=drop_columns_for_save, errors="ignore")
    .rename(columns=column_rename_map)
)

print("[저장용 User_Profile 컬럼]")
print(user_profile_save.columns.tolist())

print("\n[저장용 Event_Log 컬럼]")
print(event_log_save.columns.tolist())

[저장용 User_Profile 컬럼]
['user_id', 'signup_date', 'signup_channel', 'device', 'notification_agreed', 'notification_changed_date', 'signup_day', 'signup_month', 'signup_weekday', 'first_event_time', 'first_event_elapsed_hours', 'first_app_launch_time', 'app_launch_elapsed_hours', 'onboarding_completed_time', 'onboarding_elapsed_hours', 'app_launch_to_onboarding_hours', 'is_onboarding_completed', 'event_count_total', 'avg_events_per_session', 'is_signup_log_issue_period', 'is_no_event_user', 'is_notification_changed', 'is_first_event_before_signup']

[저장용 Event_Log 컬럼]
['user_id', 'event_time', 'event_type', 'session_id', 'notification_type', 'event_date', 'event_month', 'event_weekday', 'event_hour', 'is_log_issue_period', 'is_event_type_missing']


In [28]:
# ============================================================
# 최종 저장
# ============================================================
# 저장 폴더가 없으면 생성
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 저장할 파일 경로를 지정
user_profile_output = DATA_PROCESSED_DIR / "01_user_profile_preprocessed.csv"
event_log_output = DATA_PROCESSED_DIR / "02_event_log_preprocessed.csv"

user_profile_save.to_csv(
    user_profile_output,
    index=False,
    encoding="utf-8-sig"
)

event_log_save.to_csv(
    event_log_output,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("1.", user_profile_output)
print("2.", event_log_output)


저장 완료
1. C:\Users\milkl\OneDrive\문서\웰니스 공모전\wellness analyze\Anti-Churn-Committee\data\processed\01_user_profile_preprocessed.csv
2. C:\Users\milkl\OneDrive\문서\웰니스 공모전\wellness analyze\Anti-Churn-Committee\data\processed\02_event_log_preprocessed.csv


# 파생컬럼 사용 방법
## 01_user_profile_preprocessed.csv
| 컬럼명                              | 어디에 쓰이는지                                          |
| -------------------------------- | ------------------------------------------------- |
| `signup_day`                     | 가입일 기준으로 일자별 가입자 수를 확인할 때 사용                      |
| `signup_month`                   | 가입월별 코호트 분석이나 월별 가입자 추이를 볼 때 사용                   |
| `signup_weekday`                 | 요일별 가입 패턴이 있는지 확인할 때 사용                           |
| `first_event_time`               | 사용자가 가입 후 처음으로 앱에서 행동한 시점을 확인할 때 사용               |
| `first_event_elapsed_hours`      | 가입 후 첫 행동까지 얼마나 걸렸는지 확인해 초기 진입 속도를 볼 때 사용         |
| `first_app_launch_time`          | 사용자가 처음 앱을 실행한 시점을 확인할 때 사용                       |
| `app_launch_elapsed_hours`       | 가입 후 앱 실행까지 걸린 시간을 확인해 가입 직후 앱 접근 여부를 볼 때 사용      |
| `onboarding_completed_time`      | 사용자가 온보딩을 완료한 시점을 확인할 때 사용                        |
| `onboarding_elapsed_hours`       | 가입 후 온보딩 완료까지 걸린 시간을 확인해 온보딩 지연 여부를 볼 때 사용        |
| `app_launch_to_onboarding_hours` | 앱 실행 후 온보딩 완료까지 걸린 시간을 확인해 온보딩 과정의 이탈 가능성을 볼 때 사용 |
| `event_count_total`              | 사용자별 전체 활동량을 비교할 때 사용                             |
| `avg_events_per_session`         | 사용자가 한 세션에서 평균적으로 얼마나 행동하는지 확인할 때 사용              |

## 02_event_log_preprocessed.csv
| 컬럼명             | 어디에 쓰이는지                      |
| --------------- | ----------------------------- |
| `event_date`    | 일자별 이벤트 발생량을 볼 때 사용           |
| `event_month`   | 월별 이벤트 흐름이나 월별 사용량 변화를 볼 때 사용 |
| `event_weekday` | 요일별 사용 패턴을 확인할 때 사용           |
| `event_hour`    | 시간대별 앱 사용 패턴을 확인할 때 사용        |


# 플래그 사용 방법
## 01_user_profile_preprocessed.csv
| 컬럼명                            | 어디에 쓰이는지                             |
| ------------------------------ | ------------------------------------ |
| `is_signup_log_issue_period`   | 로그 수집 장애 기간에 가입한 사용자를 따로 구분할 때 사용    |
| `is_no_event_user`             | 프로필에는 있지만 이벤트 로그가 없는 사용자를 확인할 때 사용   |
| `is_onboarding_completed`      | 온보딩 완료자와 미완료자의 리텐션 차이를 비교할 때 사용      |
| `is_first_event_before_signup` | 가입일보다 이벤트가 먼저 찍힌 데이터 이상 여부를 확인할 때 사용 |


## 02_event_log_preprocessed.csv
| 컬럼명                     | 어디에 쓰이는지                                |
| ----------------------- | --------------------------------------- |
| `is_log_issue_period`   | 로그 수집 장애 기간의 이벤트를 구분해 분석 해석에 주의할 때 사용   |
| `is_event_type_missing` | 이벤트 종류가 비어 있는 로그를 확인해 데이터 품질 이슈를 볼 때 사용 |


# 컬럼 명세서

## `01_user_profile_preprocessed.csv`

| 명칭                               | 형식       | 예시                    | 설명                                |
| -------------------------------- | -------- | --------------------- | --------------------------------- |
| `user_id`                        | string   | `U0000001`            | 사용자 고유 ID                         |
| `signup_date`                    | datetime | `2025-01-25`          | 가입일자                              |
| `signup_channel`                 | string   | `오가닉`                 | 가입경로                              |
| `device`                         | string   | `iOS`                 | 사용 기기                             |
| `notification_agreed`            | bool     | `True`                | 알림 수신 동의 여부                       |
| `notification_changed_date`      | datetime | `2025-05-24`          | 알림 수신 동의 변경일자                     |
| `signup_day`                     | date     | `2025-01-25`          | 파생컬럼: 가입일 기준 날짜                   |
| `signup_month`                   | string   | `2025-01`             | 파생컬럼: 가입월                         |
| `signup_weekday`                 | string   | `토`                   | 파생컬럼: 가입요일                        |
| `first_event_time`               | datetime | `2025-01-25 07:25:45` | 파생컬럼: 사용자별 첫 이벤트 발생 시간            |
| `first_event_elapsed_hours`      | float    | `0.43`                | 파생컬럼: 가입 후 첫 이벤트까지 걸린 시간          |
| `first_app_launch_time`          | datetime | `2025-01-25 07:25:45` | 파생컬럼: 사용자별 첫 앱실행 시간               |
| `app_launch_elapsed_hours`       | float    | `0.43`                | 파생컬럼: 가입 후 첫 앱실행까지 걸린 시간          |
| `onboarding_completed_time`      | datetime | `2025-01-25 07:26:15` | 파생컬럼: 사용자별 첫 온보딩 완료 시간            |
| `onboarding_elapsed_hours`       | float    | `0.44`                | 파생컬럼: 가입 후 온보딩 완료까지 걸린 시간         |
| `app_launch_to_onboarding_hours` | float    | `0.01`                | 파생컬럼: 첫 앱실행 후 온보딩 완료까지 걸린 시간      |
| `event_count_total`              | int      | `35`                  | 파생컬럼: 사용자별 전체 이벤트 수               |
| `avg_events_per_session`         | float    | `3.25`                | 파생컬럼: 사용자별 세션당 평균 이벤트 수           |
| `is_signup_log_issue_period`     | bool     | `False`               | 플래그: 로그 수집 장애 기간 가입 여부            |
| `is_no_event_user`               | bool     | `False`               | 플래그: 프로필에는 있지만 이벤트 로그에는 없는 사용자 여부 |
| `is_onboarding_completed`        | bool     | `True`                | 플래그: 온보딩 완료 여부                    |
| `is_first_event_before_signup`   | bool     | `False`               | 플래그: 첫 이벤트가 가입일보다 빠른지 여부          |


## `02_event_log_preprocessed.csv`

| 명칭                      | 형식       | 예시                    | 설명                      |
| ----------------------- | -------- | --------------------- | ----------------------- |
| `user_id`               | string   | `U0000001`            | 사용자 고유 ID               |
| `event_time`            | datetime | `2025-01-25 07:25:45` | 이벤트 발생 시간               |
| `event_type`            | string   | `앱실행`                 | 이벤트 종류                  |
| `session_id`            | string   | `2858202000`          | 세션 ID, 알림 이벤트는 결측 가능    |
| `notification_type`     | string   | `광고성`                 | 알림 유형, 알림 이벤트에만 값 존재    |
| `event_date`            | date     | `2025-01-25`          | 파생컬럼: 이벤트 발생일           |
| `event_month`           | string   | `2025-01`             | 파생컬럼: 이벤트 발생월           |
| `event_weekday`         | string   | `토`                   | 파생컬럼: 이벤트 발생요일          |
| `event_hour`            | int      | `7`                   | 파생컬럼: 이벤트 발생 시간대        |
| `is_log_issue_period`   | bool     | `False`               | 플래그: 로그 수집 장애 기간 이벤트 여부 |
| `is_event_type_missing` | bool     | `False`               | 플래그: 이벤트 종류 결측 여부       |
